In [9]:
import numpy as np
import torchtune
from torchtune import utils
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path
from omegaconf import OmegaConf

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from training import SelfPredictionTrainingRecipeDistributed

In [2]:
base_dir = '/home/woody/iwbi/iwbi106h/suuraj/models/self-prediction-models'
model_path = 'learning_levels_sweep_x1bmwa36'

base_model_path = os.path.join(base_dir, model_path)
cfg = OmegaConf.load(os.path.join(base_model_path, 'config.yaml'))
cfg.checkpointer.checkpoint_dir = base_model_path
cfg.checkpointer.checkpoint_files = ["torchtune_model_last.pt"]
cfg.train_from_scratch = False
cfg.metric_logger.mode = 'disabled'

In [17]:
from dataset_classes import learning_levels_pfa_dataset
from evaluation.pfa_evaluation import process_data, compute_losses_per_level, compute_losses_per_level_statistics
import plotly.graph_objects as go
log = utils.get_logger("DEBUG")

def pfa_training_evaluation(recipe,
                            num_datapoints=500,
                            ic_generalization_evaluation=True):
    """
    Evaluates a model on PFA-based learning tasks.

    This function assesses a model's in-context learning capabilities by processing
    data from `learning_levels_pfa_dataset`. It calculates per-token losses across
    various task complexities (e.g., memorization, generalization, random sequences),
    generates summary plots, and computes key metrics.

    The primary metric is the "interestingness ratio," which compares the model's
    loss on complex, generative tasks versus simple, memorization-based tasks.
    Optionally, it can also evaluate the model's ability to generate syntactically
    correct sequences.

    Args:
        recipe (Any): A recipe object containing the model (`_model`), tokenizer,
            and configuration.
        num_datapoints (int, optional): The number of data points to process for
            calculating losses. Defaults to 500.
        ic_generalization_evaluation (bool, optional): If True, runs an additional
            evaluation to measure the length of correctly generated PFA sequences.
            Defaults to True.

    Returns:
        Tuple[Dict[str, go.Figure], Dict[str, float]]: A tuple containing:
        - A dictionary mapping loss names (e.g., "phi_losses") to Plotly bar
          chart figures visualizing the loss per learning level.
        - A dictionary of scalar evaluation metrics, including
          "interestingness_ratio" and optionally "length_generalization".
    """
    recipe._model.eval()

    kwargs = dict(recipe.cfg.dataset)
    kwargs.pop("_component_")
    kwargs["tokenizer"] = recipe._tokenizer
    dataset = learning_levels_pfa_dataset(**kwargs)

    datapoints = process_data(
        recipe,
        dataset=dataset,
        num_datapoints=num_datapoints,
    )
    losses = ["next_token_losses"]
    interestingness_criterion = "next_token_losses"

    losses.append("phi_losses",)
    losses.append("latent_losses",)

    log.info(datapoints[0].keys())
    
    # if "phi_losses" in datapoints[0]:
    #     losses.append("phi_losses")
    #     interestingness_criterion = "phi_losses"

    level_names = [
        "memorized sequence",
        "memorized language",
        "learned vocabulary",
        "learned language",
        "random",
        "copy",
    ]
    levels = tuple(dataset.included_learning_levels)
    interesting_levels = (2, 3)
    uninteresting_levels = (0, 1, 4, 5)
    # find the intersection between both interesting and uninteresting levels and the levels in the dataset
    interesting_levels = list(set(interesting_levels).intersection(set(levels)))
    uninteresting_levels = list(set(uninteresting_levels).intersection(set(levels)))

    losses_vs_learning_levels = compute_losses_per_level(
        datapoints,
        filter_out_spaces=False,
        losses=losses,
        levels=levels
    )
    losses_vs_learning_levels_statistics = compute_losses_per_level_statistics(
        losses_vs_learning_levels,
        losses=losses,
        levels=levels
    )

    if ic_generalization_evaluation:
        length_generalization_results = evaluate_language_generation_length(recipe, num_samples=500)
        length_generalization = length_generalization_results['mean']
    else:
        length_generalization = None

    plotly_figure_dict = {}
    interestingness_ratio = 0.0

    group_losses = [[]]
    # group_losses = [
    #     ["prediction_entropy", "target_entropy"],
    #     ]

    loss_filter = [False for _ in losses]
    for i, loss in enumerate(losses):
        for group in group_losses:
            if loss in group:
                loss_filter[i] = True

    for i, loss in enumerate(losses):
        if loss_filter[i]:
            continue
        means = [
            losses_vs_learning_levels_statistics[loss][level]["mean"]
            for level in levels
        ]
        if loss == interestingness_criterion:
            if len(interesting_levels) > 0 and len(uninteresting_levels) > 0:
                interesting_means = [means[level] for level in interesting_levels]
                uninteresting_means = [means[level] for level in uninteresting_levels]
                interestingness_ratio = np.mean(interesting_means) / np.mean(
                    uninteresting_means
                )
        fig = go.Figure(data=[go.Bar(x=levels, y=means)])
        # add level names
        fig.update_layout(
            title=f"{loss.capitalize()}",
            xaxis_title="Learning level",
            yaxis_title="",
            xaxis=dict(tickvals=levels, ticktext=[level_names[l] for l in levels]),
        )
        plotly_figure_dict[loss] = fig

    for group in group_losses:
        if group == []:
            continue
        group_means = []
        fig = go.Figure()
        for loss in group:
            means = [
                losses_vs_learning_levels_statistics[loss][level]["mean"]
                for level in levels
            ]
            group_means.append(means)
            if loss == interestingness_criterion:
                if len(interesting_levels) > 0 and len(uninteresting_levels) > 0:
                    interesting_means = [means[level] for level in interesting_levels]
                    uninteresting_means = [means[level] for level in uninteresting_levels]
                    interestingness_ratio = np.mean(interesting_means) / np.mean(
                        uninteresting_means
                    )
            fig.add_trace(go.Bar(x=levels, y=means, name=loss.capitalize() ))
        # add level names
        fig.update_layout(
            title=f"{loss.capitalize()}",
            xaxis_title="Learning level",
            yaxis_title="",
            xaxis=dict(tickvals=levels, ticktext=[level_names[l] for l in levels]),
        )
        plotly_figure_dict[loss] = fig



    recipe._model.train()
    eval_values_dict = {
        "interestingness_ratio": interestingness_ratio,
    }
    if length_generalization is not None:
        eval_values_dict["length_generalization"] = length_generalization
    recipe._model.self_prediction_losses.reset()
    return datapoints, plotly_figure_dict, eval_values_dict

In [4]:
recipe = SelfPredictionTrainingRecipeDistributed(cfg=cfg)
recipe.setup(cfg=cfg)

DEBUG:torchtune.utils._logging:Setting manual seed to local seed 92075773. Local seed is seed + rank = 92075773 + 0
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
INFO:torchtune.utils._logging:Model is initialized with precision torch.bfloat16.
INFO:torchtune.utils._logging:Memory stats after model init:
	GPU peak memory allocation: 0.18 GiB
	GPU peak memory reserved: 0.20 GiB
	GPU peak memory active: 0.18 GiB
INFO:torchtune.utils._logging:Optimizer is initialized.
INFO:torchtune.utils._logging:information bottleneck: continuous
INFO:torchtune.utils._logging:phi loss factor: 0.001
INFO:torchtune.utils._logging:self critic loss factor: 0.1
INFO:torchtune.utils._logging:Loss is initialized.


run id:  zgxbzer4
None


INFO:torchtune.utils._logging:Dataset and Sampler are initialized.
INFO:torchtune.utils._logging:Learning rate scheduler is initialized.
INFO:torchtune.utils._logging: Profiler config after instantiation: {'enabled': False}


In [19]:
recipe._model.eval()

datapoints, _, eval_dict = pfa_training_evaluation(recipe,num_datapoints = 20,ic_generalization_evaluation=False)
datapoints

Processing batch 1/20
Processing batch 5/20
Processing batch 9/20
Processing batch 13/20
Processing batch 17/20


INFO:torchtune.utils._logging:dict_keys(['tokens', 'labels', 'input_pos', 'new_language', 'learning_level', 'num_states', 'num_edges', 'vocab_size', 'perturbation', 'next_token_losses', 'latent_losses', 'latent_entropy', 'phi_losses'])


[{'tokens': array([  2, 106, 101, ..., 114,  32,   3]),
  'labels': array([  2, 106, 101, ..., 114,  32,   3]),
  'input_pos': array([   0,    1,    2, ..., 1759, 1760, 1761]),
  'new_language': array([0, 1, 0, ..., 0, 0, 0]),
  'learning_level': array([-1,  2,  2, ...,  3,  3, -1]),
  'num_states': array([-1,  8,  8, ...,  9,  9, -1]),
  'num_edges': array([-1, 15, 15, ..., 19, 19, -1]),
  'vocab_size': array([-1,  6,  6, ..., 13, 13, -1]),
  'perturbation': array([0, 0, 0, ..., 0, 0, 0]),
  'next_token_losses': array([0.       , 2.703125 , 2.609375 , ..., 1.0078125, 1.4921875,
         3.671875 ], dtype=float32),
  'latent_losses': array([6336.,  920., 1072., ...,  724.,  856.,  572.], dtype=float32),
  'latent_entropy': array([1012., 1376., 1400., ..., 1328., 1344., 1392.], dtype=float32),
  'phi_losses': array([36.25, 36.75, 53.5 , ..., 25.5 , 38.  , 38.  ], dtype=float32)},
 {'tokens': array([  2, 101, 114, ..., 101,  32,   3]),
  'labels': array([  2, 101, 114, ..., 101,  32,   3